In [5]:
import rasterio
from rasterio.windows import Window
from rasterio.env import Env
import numpy as np
import psutil
import os

red_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B04_10m.jp2"
nir_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B08_10m.jp2"
tile_size = 512

process = psutil.Process(os.getpid())
with Env(GDAL_CACHEMAX=128): 
    
    with rasterio.open(red_path) as red_src, rasterio.open(nir_path) as nir_src:

        height = red_src.height
        width = red_src.width

        print(f"Image size: {width} x {height}")
        for row in range(0,height,tile_size):
            for col in range(0,width,tile_size):

                win_h=min(tile_size , height-row)
                win_w= min(tile_size , width - col)

                window = Window(col , row , win_w , win_h)
                red = red_src.read(1,window=window).astype("float32")/10000.0
                nir = nir_src.read(1,window= window).astype("float32")/10000.0
                if np.all(red==0) and np.all(nir==0):
                    continue
                denominator = nir + red
                denominator[denominator == 0] = 1e-6

                ndvi = (nir - red) / denominator 
                mean = np.mean(ndvi)
                print(f"The mean NDVI is {mean:4f}")

                mem = process.memory_info().rss / (1024**2)
                print(f"The memory usage is {mem:2f} MB")

                del red
                del nir
                del ndvi


Image size: 10980 x 10980
The mean NDVI is 0.362279
The memory usage is 119.503906 MB
The mean NDVI is 0.383275
The memory usage is 119.863281 MB
The mean NDVI is 0.362391
The memory usage is 119.941406 MB
The mean NDVI is 0.342465
The memory usage is 120.046875 MB
The mean NDVI is 0.284173
The memory usage is 120.394531 MB
The mean NDVI is 0.245403
The memory usage is 120.050781 MB
The mean NDVI is 0.259515
The memory usage is 120.375000 MB
The mean NDVI is 0.274612
The memory usage is 120.437500 MB
The mean NDVI is 0.046086
The memory usage is 120.492188 MB
The mean NDVI is 0.381535
The memory usage is 120.570312 MB
The mean NDVI is 0.287672
The memory usage is 120.312500 MB
The mean NDVI is 0.288282
The memory usage is 120.550781 MB
The mean NDVI is 0.266294
The memory usage is 120.710938 MB
The mean NDVI is 0.239809
The memory usage is 120.656250 MB
The mean NDVI is 0.232233
The memory usage is 121.238281 MB
The mean NDVI is 0.235843
The memory usage is 121.191406 MB
The mean NDVI 

In [14]:
import rasterio
from rasterio.windows import Window
from rasterio.env import Env
import numpy as np
import time
import psutil
import os

red_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B04_10m.jp2"
nir_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B08_10m.jp2"
tile_size = 1024
tile_count = 0
max_ndvi = -1.0
latitude, longitude = 0, 0

process = psutil.Process(os.getpid())
start_time_ = time.time()

with Env(GDAL_CACHEMAX=128):
    with rasterio.open(red_path) as red_src,rasterio.open(nir_path) as nir_src:

        height, width = red_src.height, red_src.width

        for row in range(0, height, tile_size):
            for col in range(0, width, tile_size):

                win_h = min(tile_size, height - row)
                win_w = min(tile_size, width - col)

                window = Window(col, row, win_w, win_h)

                red = red_src.read(1, window=window).astype("float32") / 10000.0
                nir = nir_src.read(1, window=window).astype("float32") / 10000.0

                if np.all(red == 0) and np.all(nir == 0):
                    continue

                denom = nir + red
                denom[denom == 0] = 1e-6

                ndvi = (nir - red) / denom
                mean_ndvi = np.mean(ndvi)
                if mean_ndvi>0.2:
                    continue

                transform = red_src.window_transform(window)
                lon, lat = transform * (win_w // 2, win_h // 2)

                if mean_ndvi > max_ndvi:
                    max_ndvi = mean_ndvi
                    latitude, longitude = lat, lon

                if tile_count % 50 == 0:
                    mem = process.memory_info().rss / 1024**2
                    print(f"[{tile_count}] NDVI: {mean_ndvi:.4f}, Mem: {mem:.2f} MB")

                tile_count += 1

                del red, nir, ndvi

elapsed_time = time.time() - start_time_

print("\n-------------------------------")
print(f"Tiles processed = {tile_count}")
print(f"Total time = {elapsed_time:.2f} sec")
print(f"Speed = {tile_count / elapsed_time:.2f} tiles/sec")
print(f"Max NDVI at lat={latitude}, lon={longitude}")

[0] NDVI: 0.0120, Mem: 146.46 MB

-------------------------------
Tiles processed = 20
Total time = 4.80 sec
Speed = 4.17 tiles/sec
Max NDVI at lat=1933480.0, lon=705080.0
